# Notebook de orquestación

En la siguiente celda se ejecuta el script de descarga y limpieza de los datos directamente del sitio web:

(https://insideairbnb.com/get-the-data/)

Se sigue un proceso ETL

In [ ]:
from download import run

dl = run(url="s3://airbnb-pred/airbnb-price/listings/listings.csv.gz")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


2026-05-22 22:49:30 | INFO     | download:download | Descarga


2026-05-22 22:49:30 | INFO     | download:download | Leyendo desde S3: s3://airbnb-pred/airbnb-price/listings/listings.csv.gz


2026-05-22 22:49:32 | INFO     | download:download | 27,051 filas  |  79 columnas


2026-05-22 22:49:32 | INFO     | download:download | Preprocesamiento general


2026-05-22 22:49:32 | INFO     | download:download | raw_data_entire:  15,647 filas


2026-05-22 22:49:32 | INFO     | download:download | raw_data_private: 6,627 filas


2026-05-22 22:49:32 | INFO     | download:download | raw_data_final:   22,274 filas


2026-05-22 22:49:32 | INFO     | download:download | Guardado como 'airbnb_final2.parquet'


Instalación de versión de Sagemaker que no causa conflictos.

In [ ]:
# Instalar versión de sagemaker
%pip install 'sagemaker==2.257.3' --quiet

Note: you may need to restart the kernel to use updated packages.


En la siguiente celda se importan las bibliotecas principales. Esta celda se debe correr primero.

In [ ]:
import sys, os
import json
import sagemaker, boto3
from sagemaker.estimator import Estimator
import config
import boto3

print(f'SageMaker version : {sagemaker.__version__}')
print(f'Bucket            : {config.BUCKET_NAME}')
print(f'Región            : {config.REGION}')
print(f'Rol IAM           : {config.ROLE_ARN}')
print(f'Endpoint          : {config.ENDPOINT_NAME}')
print(f'CW Log Group      : {config.CW_LOG_GROUP}')
print(f'Random Search     : {config.RS_N_ITER} iter × {config.RS_CV_FOLDS} folds')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


SageMaker version : 2.257.3
Bucket            : airbnb-pred
Región            : us-east-1
Rol IAM           : arn:aws:iam::141095608224:role/SageMakerStudioExecutionRole2026
Endpoint          : airbnb-v2-xgboost
CW Log Group      : /airbnb-price-prediction/v2
Random Search     : 50 iter × 5 folds


La siguiente celda sube el parquet con los datos crudos a un bucket previamente creado manualmente de S3. Aquí termina el proceso ETL. Ese parquet generado vive en la capa Gold.

In [ ]:
LOCAL_PARQUET = 'airbnb_final.parquet'
s3 = boto3.client('s3', region_name=config.REGION)
s3.upload_file(
    LOCAL_PARQUET,
    config.BUCKET_NAME,
    f'{config.PREFIX}/raw/{LOCAL_PARQUET}',
)

### Preprocesamiento
En la siguiente celda se aplica el proceso de preprocesamiento de los datos crudos.

In [ ]:
from preprocessing import run_preprocessing

train_uri, val_uri, encodings, rs_results = run_preprocessing(run_rs=True)

best_params = rs_results.get('best_params', {})
print('Mejores hiperparámetros encontrados:')
for k, v in sorted(best_params.items()):
    print(f'   {k:<22}: {v}')
print(f'   CV RMSE (log)  : {rs_results.get("cv_rmse", "N/A")}')
print(f'   Val R²         : {rs_results.get("val_r2", "N/A")}')

2026-05-22 06:29:27 | INFO     | preprocessing:preprocessing | Inicio del pipeline de preprocesamiento


2026-05-22 06:29:27 | INFO     | preprocessing:preprocessing | Configuración: smoothing=10, oversample_min=200


2026-05-22 06:29:27 | INFO     | preprocessing:preprocessing | Leyendo dataset desde S3 s3://airbnb-pred/airbnb-price/raw/airbnb_final.parquet


2026-05-22 06:29:27 | INFO     | preprocessing:preprocessing | Dataset cargado: 22,274 filas, 11 columnas


2026-05-22 06:29:27 | INFO     | preprocessing:preprocessing | Iniciando feature engineering


...

2026-05-22 06:32:02 | INFO     | preprocessing:preprocessing | Split arriba: s3://airbnb-pred/airbnb-price/data/validation/validation.csv


2026-05-22 06:32:02 | INFO     | preprocessing:preprocessing | Preprocesamiento completo


2026-05-22 06:32:02 | INFO     | preprocessing:preprocessing |    Train : s3://airbnb-pred/airbnb-price/data/train/train.csv


2026-05-22 06:32:02 | INFO     | preprocessing:preprocessing |    Val   : s3://airbnb-pred/airbnb-price/data/validation/validation.csv


Mejores hiperparámetros encontrados:
   colsample_bytree      : 1.0
   gamma                 : 0
   learning_rate         : 0.1
   max_depth             : 8
   min_child_weight      : 3
   n_estimators          : 200
   reg_alpha             : 0.5
   reg_lambda            : 2.0
   subsample             : 0.9
   CV RMSE (log)  : 0.42871005839361975
   Val R²         : 0.66858491415553


## Entrenamiento

La siguiente celda corre el entrenamiento

In [ ]:
from train import run_training

estimator = run_training(
    train_uri   = train_uri,
    val_uri     = val_uri,
    best_params = best_params,
)

2026-05-22 06:32:51 | INFO     | train:training | Inicio del training job


2026-05-22 06:32:51 | INFO     | train:training | Hiperparámetros para SageMaker {"subsample": 0.9, "lambda": 2.0, "alpha": 0.5, "num_round": 200, "min_child_weight": 3, "max_depth": 8, "eta": 0.1, "gamma": 0, "colsample_bytree": 1.0, "objective": "reg:squarederror", "eval_metric": "rmse", "early_stopping_rounds": 30}


2026-05-22 06:32:51 | INFO     | train:training | Imagen XGBoost: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


2026-05-22 06:32:51 | INFO     | train:training | Instancia: ml.m5.xlarge


2026-05-22 06:32:51 | INFO     | train:training | Train URI: s3://airbnb-pred/airbnb-price/data/train/train.csv


...

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-22 06:34:58.865 ip-10-0-88-42.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-05-22 06:34:58.928 ip-10-0-88-42.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-05-22:06:34:59:INFO] Imported framework sagemaker_xgboost_container.training
[2026-05-22:06:34:59:INFO] Failed to parse hyperparameter eval_metric value rmse to Json.
Returning the value itself
[2026-05-22:06:34:59:INFO] Failed to parse hyperparameter objective value reg:squarederror to Json.
Returning the value itself
[2026-05-22:06:34:59:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-22:06:34:59:INFO] Runni


2026-05-22 06:35:21 Uploading - Uploading generated training model
2026-05-22 06:35:21 Completed - Training job completed


2026-05-22 06:35:39 | INFO     | train:training | Training completado Job: airbnb-v2-xgb-2026-05-22-06-32-51-724


Training seconds: 109
Billable seconds: 109


2026-05-22 06:35:39 | INFO     | train:training | Artefacto modelo s3://airbnb-pred/airbnb-price/model/airbnb-v2-xgb-2026-05-22-06-32-51-724/output/model.tar.gz


In [ ]:
from train import list_training_jobs
list_training_jobs()

2026-05-22 06:37:23 | INFO     | train:training | Job: airbnb-v2-xgb-2026-05-22-06-32-51-724  Status: Completed


2026-05-22 06:37:23 | INFO     | train:training | Job: airbnb-v2-xgb-2026-05-21-23-49-13-632  Status: Completed


2026-05-22 06:37:23 | INFO     | train:training | Job: airbnb-v2-xgb-2026-05-17-20-42-32-621  Status: Completed


## Inferencia

La siguiente celda crea el endpoint usando el último training job creado.

In [ ]:
from train import deploy_endpoint
from cw_logger import get_logger

logger = get_logger(__name__, stream_name="endpoint")
sm = boto3.client("sagemaker", region_name=config.REGION)

latest_job = sm.list_training_jobs(
    NameContains = config.JOB_BASE_NAME,
    MaxResults   = 1,
    SortBy       = "CreationTime",
    SortOrder    = "Descending",
)["TrainingJobSummaries"][0]

# comprueba si hay configuración del endpoint anterior
try:
    sm.delete_endpoint_config(EndpointConfigName=config.ENDPOINT_NAME)
    logger.info(f"Endpoint config eliminado: {config.ENDPOINT_NAME}")
except sm.exceptions.ClientError:
    print("Endpoint config no existía")

# comprueba si el endpoint está arriba
try:
    sm.delete_endpoint(EndpointName=config.ENDPOINT_NAME)
    logger.info(f"Endpoint eliminado: {config.ENDPOINT_NAME}")
    
    # Espera a que termine de eliminarse
    waiter = sm.get_waiter("endpoint_deleted")
    waiter.wait(
        EndpointName = config.ENDPOINT_NAME,
        WaiterConfig = {"Delay": 10, "MaxAttempts": 30},
    )
    logger.info("Endpoint completamente eliminado")
except sm.exceptions.ClientError:
    logger.info("Endpoint no existía, solo se eliminó el config")

# sesión de sagemaker
session = sagemaker.Session(
    boto_session=boto3.Session(region_name=config.REGION)
)

# le añade el training job al estimator
estimator = Estimator.attach(
    training_job_name = latest_job["TrainingJobName"],
    sagemaker_session = session,
)

predictor = deploy_endpoint(estimator)
logger.info("El endpoint ha sido deployado")

2026-05-22 06:58:39 | INFO     | __main__:endpoint | Endpoint config eliminado: airbnb-v2-xgboost


2026-05-22 06:58:40 | INFO     | __main__:endpoint | Endpoint eliminado: airbnb-v2-xgboost


2026-05-22 06:58:50 | INFO     | __main__:endpoint | Endpoint completamente eliminado



2026-05-22 06:35:21 Starting - Preparing the instances for training
2026-05-22 06:35:21 Downloading - Downloading the training image
2026-05-22 06:35:21 Training - Training image download completed. Training in progress.
2026-05-22 06:35:21 Uploading - Uploading generated training model
2026-05-22 06:35:21 Completed - Training job completed

2026-05-22 06:58:56 | INFO     | train:training | Desplegando endpoint: airbnb-v2-xgboost


...

-

-

!

2026-05-22 07:02:28 | INFO     | train:training | Endpoint activo: airbnb-v2-xgboost


2026-05-22 07:02:28 | INFO     | __main__:endpoint | El endpoint ha sido deployado


## Inferencia directa (sin LLM)
En la siguiente celda se prueba que el endpoint haga inferencia en tiempo real.

In [ ]:
from inference import AirbnbPredictor
import pandas as pd

airbnb = AirbnbPredictor()

precio = airbnb.predict(
    neighbourhood = 'Cuauhtémoc',
    room_type     = 'Private room',
    latitude      = 19.4216,
    longitude     = -99.1623,
    accommodates  = 4,
    bathrooms     = 1.0,
    bedrooms      = 1,
    beds          = 2,
    parking       = 1,
    patio_balcon  = 0,
)
print(f'Precio estimado: ${precio:,.2f} MXN / noche')

2026-05-22 07:12:47 | INFO     | inference:inference | Encodings cargados desde s3://airbnb-pred/airbnb-price/artifacts/encodings.json


2026-05-22 07:12:47 | INFO     | inference:inference | AirbnbPredictor inicializado. Endpoint: airbnb-v2-xgboost


2026-05-22 07:12:47 | INFO     | inference:inference | Predicción: neighbourhood=Cuauhtémoc, room_type=Private room, accommodates=4, bedrooms=1


2026-05-22 07:12:47 | INFO     | inference:inference | Precio estimado: $993.08 MXN


Precio estimado: $993.08 MXN / noche


## Inferencia con lenguaje natural vía Amazon Bedrock

**Restricciones activas:**
1. Ubicación debe ser CDMX (alcaldía, colonia, dirección, zona)
2. Defaults si falta info: 2 ocupantes, 2 recámaras, 2 camas, 1 baño, sin parking, sin patio
3. beds = bedrooms si no se especifican camas
4. Dirección/colonia sin alcaldía → geocodificación con Bedrock
5. Texto sin relación con alojamientos → mensaje de error

In [ ]:
from bedrock import AirbnbBedrockPipeline

pipeline = AirbnbBedrockPipeline(
    endpoint_name  = config.ENDPOINT_NAME,
    region         = config.REGION,
)

2026-05-22 07:10:52 | INFO     | bedrock:bedrock | Caché de geocodificación cargado: 3 entradas


2026-05-22 07:10:52 | INFO     | bedrock:bedrock | Pipeline inicializado | endpoint=airbnb-v2-xgboost | model=us.anthropic.claude-haiku-4-5-20251001-v1:0


Pipeline listo.
Bedrock model : us.anthropic.claude-haiku-4-5-20251001-v1:0
SM endpoint   : airbnb-v2-xgboost


En las siguientes celdas se corren ejemplo para probar la inferencia con lenguaje natural.

In [ ]:
pipeline.predict_from_text(
    'Tengo un departamento en Coyoacán para 4 personas, '
    '2 recámaras, 1 baño, tiene balcón pero no estacionamiento.'
)

2026-05-22 07:15:30 | INFO     | bedrock:bedrock | Solicitud recibida: 'Tengo un departamento en Coyoacán para 4 personas, 2 recámaras, 1 baño, tiene balcón pero no estacio'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:15:31 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Coyoacán", "room_type": "Entire home/apt", "accommodates": 4, "bedrooms": 2, "beds": null, "bathrooms": 1.0, "parking": 0, "patio_balcon": 1}


2026-05-22 07:15:31 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Coyoacán", "room_type": "Entire home/apt", "latitude": 19.3391, "longitude": -99.1585, "accommodates": 4, "bedrooms": 2, "beds": 2, "bathrooms": 1.0, "parking": 0, "patio_balcon": 1}


2026-05-22 07:15:32 | INFO     | bedrock:bedrock | Predicción exitosa: $1,152.51 MXN | neighbourhood=Coyoacán | room_type=Entire home/apt


   JSON extraído: {"is_valid_listing": true, "location": "Coyoacán", "room_type": "Entire home/apt", "accommodates": 4, "bedrooms": 2, "beds": null, "bathrooms": 1.0, "parking": 0, "patio_balcon": 1}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Coyoacán  [alias]
  Tipo de alojamiento   : Entire home/apt
  Latitud               : 19.3391
  Longitud              : -99.1585
  Ocupantes             : 4
  Habitaciones          : 2
  Camas                 : 2
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✓ Sí
──────────────────────────────────────────────────────
Precio estimado : $    1,152.51 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Coyoacán',
  'room_type': 'Entire home/apt',
  'latitude': 19.3391,
  'longitude': -99.1585,
  'accommodates': 4,
  'bedrooms': 2,
  'beds': 2,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 1},
 'price': 1152.51}

In [ ]:
pipeline.predict_from_text(
    'Rento mi cuarto privado en la Condesa, para 2 personas, cama individual, baño compartido.'
)

2026-05-22 07:15:35 | INFO     | bedrock:bedrock | Solicitud recibida: 'Rento mi cuarto privado en la Condesa, para 2 personas, cama individual, baño compartido.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:15:36 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Condesa", "room_type": "Private room", "accommodates": 2, "bedrooms": 1, "beds": 1, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:15:36 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Cuauhtémoc", "room_type": "Private room", "latitude": 19.4216, "longitude": -99.1623, "accommodates": 2, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:15:36 | INFO     | bedrock:bedrock | Predicción exitosa: $884.78 MXN | neighbourhood=Cuauhtémoc | room_type=Private room


   JSON extraído: {"is_valid_listing": true, "location": "Condesa", "room_type": "Private room", "accommodates": 2, "bedrooms": 1, "beds": 1, "bathrooms": null, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Cuauhtémoc  [alias]
  Tipo de alojamiento   : Private room
  Latitud               : 19.4216
  Longitud              : -99.1623
  Ocupantes             : 2
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      884.78 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Cuauhtémoc',
  'room_type': 'Private room',
  'latitude': 19.4216,
  'longitude': -99.1623,
  'accommodates': 2,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 884.78}

In [ ]:
pipeline.predict_from_text(
    'Rento mi cuarto privado en la alcaldía Milpa Alta'
    'cama individual, baño compartido.'
)

2026-05-22 07:15:40 | INFO     | bedrock:bedrock | Solicitud recibida: 'Rento mi cuarto privado en la alcaldía Milpa Altacama individual, baño compartido.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:15:41 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Milpa Altacama", "room_type": "Private room", "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:15:41 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Milpa Alta", "room_type": "Private room", "latitude": 19.1981, "longitude": -99.0473, "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:15:41 | INFO     | bedrock:bedrock | Predicción exitosa: $433.76 MXN | neighbourhood=Milpa Alta | room_type=Private room


   JSON extraído: {"is_valid_listing": true, "location": "Milpa Altacama", "room_type": "Private room", "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": null, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Milpa Alta  [alias]
  Tipo de alojamiento   : Private room
  Latitud               : 19.1981
  Longitud              : -99.0473
  Ocupantes             : 1
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      433.76 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Milpa Alta',
  'room_type': 'Private room',
  'latitude': 19.1981,
  'longitude': -99.0473,
  'accommodates': 1,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 433.76}

In [ ]:
pipeline.predict_from_text(
    'Rento mi cuarto privado en la alcaldía Cuauhtemoc'
    'cama individual, baño compartido.'
)

2026-05-22 07:15:49 | INFO     | bedrock:bedrock | Solicitud recibida: 'Rento mi cuarto privado en la alcaldía Cuauhtemoccama individual, baño compartido.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:15:50 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Cuauhtemoc", "room_type": "Private room", "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:15:50 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Cuauhtémoc", "room_type": "Private room", "latitude": 19.4216, "longitude": -99.1623, "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:15:50 | INFO     | bedrock:bedrock | Predicción exitosa: $539.91 MXN | neighbourhood=Cuauhtémoc | room_type=Private room


   JSON extraído: {"is_valid_listing": true, "location": "Cuauhtemoc", "room_type": "Private room", "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": null, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Cuauhtémoc  [alias]
  Tipo de alojamiento   : Private room
  Latitud               : 19.4216
  Longitud              : -99.1623
  Ocupantes             : 1
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      539.91 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Cuauhtémoc',
  'room_type': 'Private room',
  'latitude': 19.4216,
  'longitude': -99.1623,
  'accommodates': 1,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 539.91}

In [ ]:
pipeline.predict_from_text(
    'Rento mi cuarto privado en la alcaldía Iztacalco'
    'cama individual, baño compartido.'
)

2026-05-22 07:15:56 | INFO     | bedrock:bedrock | Solicitud recibida: 'Rento mi cuarto privado en la alcaldía Iztacalcocama individual, baño compartido.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:15:58 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Iztacalco", "room_type": "Private room", "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": null, "patio_balcon": null}


2026-05-22 07:15:58 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Iztacalco", "room_type": "Private room", "latitude": 19.4006, "longitude": -99.0946, "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:15:58 | INFO     | bedrock:bedrock | Predicción exitosa: $737.55 MXN | neighbourhood=Iztacalco | room_type=Private room


   JSON extraído: {"is_valid_listing": true, "location": "Iztacalco", "room_type": "Private room", "accommodates": 1, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Iztacalco  [alias]
  Tipo de alojamiento   : Private room
  Latitud               : 19.4006
  Longitud              : -99.0946
  Ocupantes             : 1
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      737.55 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Iztacalco',
  'room_type': 'Private room',
  'latitude': 19.4006,
  'longitude': -99.0946,
  'accommodates': 1,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 737.55}

In [ ]:
pipeline.predict_from_text(
    'Cerro del Vigilante 91, para 2 personas, 1 recámara.'
)

2026-05-22 07:16:03 | INFO     | bedrock:bedrock | Solicitud recibida: 'Cerro del Vigilante 91, para 2 personas, 1 recámara.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:16:04 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Cerro del Vigilante 91", "room_type": "Entire home/apt", "accommodates": 2, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:16:04 | INFO     | bedrock:bedrock | Caché HIT para 'Cerro del Vigilante 91' (match='Cerro del Vigilante 91', score=1.00): Coyoacán (lat=19.3350, lon=-99.1620)


2026-05-22 07:16:04 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Coyoacán", "room_type": "Entire home/apt", "latitude": 19.335, "longitude": -99.162, "accommodates": 2, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:16:04 | INFO     | bedrock:bedrock | Predicción exitosa: $645.56 MXN | neighbourhood=Coyoacán | room_type=Entire home/apt


   JSON extraído: {"is_valid_listing": true, "location": "Cerro del Vigilante 91", "room_type": "Entire home/apt", "accommodates": 2, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
Caché HIT [exacto]: 'Cerro del Vigilante 91' → 'Cerro del Vigilante 91' → Coyoacán (lat=19.3350, lon=-99.1620)
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Coyoacán  [geocodificado por Bedrock]
  Tipo de alojamiento   : Entire home/apt
  Latitud               : 19.3350
  Longitud              : -99.1620
  Ocupantes             : 2
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      645.56 MXN / noche
────

{'fields': {'neighbourhood': 'Coyoacán',
  'room_type': 'Entire home/apt',
  'latitude': 19.335,
  'longitude': -99.162,
  'accommodates': 2,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 645.56}

In [ ]:
pipeline.predict_from_text(
    'Rento un estudio cerca del metro Polanco.'
)

2026-05-22 07:16:13 | INFO     | bedrock:bedrock | Solicitud recibida: 'Rento un estudio cerca del metro Polanco.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:16:14 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Metro Polanco", "room_type": "Entire home/apt", "accommodates": null, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:16:14 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Miguel Hidalgo", "room_type": "Entire home/apt", "latitude": 19.432, "longitude": -99.1914, "accommodates": 2, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:16:14 | INFO     | bedrock:bedrock | Predicción exitosa: $992.62 MXN | neighbourhood=Miguel Hidalgo | room_type=Entire home/apt


   JSON extraído: {"is_valid_listing": true, "location": "Metro Polanco", "room_type": "Entire home/apt", "accommodates": null, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Miguel Hidalgo  [geocodificado por Bedrock]
  Tipo de alojamiento   : Entire home/apt
  Latitud               : 19.4320
  Longitud              : -99.1914
  Ocupantes             : 2
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      992.62 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Miguel Hidalgo',
  'room_type': 'Entire home/apt',
  'latitude': 19.432,
  'longitude': -99.1914,
  'accommodates': 2,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 992.62}

In [ ]:
pipeline.predict_from_text(
    'Tengo una casa en Guadalajara, 3 habitaciones, 2 baños.'
)

2026-05-22 07:16:21 | INFO     | bedrock:bedrock | Solicitud recibida: 'Tengo una casa en Guadalajara, 3 habitaciones, 2 baños.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:16:22 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": false, "location": null, "room_type": null, "accommodates": null, "bedrooms": null, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:16:22 | WARNING  | bedrock:bedrock | Solicitud inválida: 'Tengo una casa en Guadalajara, 3 habitaciones, 2 baños.'


   JSON extraído: {"is_valid_listing": false, "location": null, "room_type": null, "accommodates": null, "bedrooms": null, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}
Por favor haz una solicitud válida relacionada con un alojamiento.


'Por favor haz una solicitud válida relacionada con un alojamiento.'

In [ ]:
pipeline.predict_from_text('hola, me siento triste')

2026-05-22 07:16:34 | INFO     | bedrock:bedrock | Solicitud recibida: 'hola, me siento triste'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:16:35 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": false, "location": null, "room_type": null, "accommodates": null, "bedrooms": null, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:16:35 | WARNING  | bedrock:bedrock | Solicitud inválida: 'hola, me siento triste'


   JSON extraído: {"is_valid_listing": false, "location": null, "room_type": null, "accommodates": null, "bedrooms": null, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}
Por favor haz una solicitud válida relacionada con un alojamiento.


'Por favor haz una solicitud válida relacionada con un alojamiento.'

In [ ]:
pipeline.predict_from_text('tengo una habitación privada en iztacalco')

2026-05-22 07:16:40 | INFO     | bedrock:bedrock | Solicitud recibida: 'tengo una habitación privada en iztacalco'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:16:41 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Iztacalco", "room_type": "Private room", "accommodates": null, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:16:41 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Iztacalco", "room_type": "Private room", "latitude": 19.4006, "longitude": -99.0946, "accommodates": 2, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:16:41 | INFO     | bedrock:bedrock | Predicción exitosa: $965.12 MXN | neighbourhood=Iztacalco | room_type=Private room


   JSON extraído: {"is_valid_listing": true, "location": "Iztacalco", "room_type": "Private room", "accommodates": null, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Iztacalco  [alias]
  Tipo de alojamiento   : Private room
  Latitud               : 19.4006
  Longitud              : -99.0946
  Ocupantes             : 2
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      965.12 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Iztacalco',
  'room_type': 'Private room',
  'latitude': 19.4006,
  'longitude': -99.0946,
  'accommodates': 2,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 965.12}

In [ ]:
pipeline.predict_from_text('tengo una habitación en mi departamento en Iztapalapa')

2026-05-22 07:17:41 | INFO     | bedrock:bedrock | Solicitud recibida: 'tengo una habitación en mi departamento en Iztapalapa'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:17:42 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Iztapalapa", "room_type": "Private room", "accommodates": null, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}


2026-05-22 07:17:42 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Iztapalapa", "room_type": "Private room", "latitude": 19.3566, "longitude": -99.0801, "accommodates": 2, "bedrooms": 1, "beds": 1, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:17:42 | INFO     | bedrock:bedrock | Predicción exitosa: $438.64 MXN | neighbourhood=Iztapalapa | room_type=Private room


   JSON extraído: {"is_valid_listing": true, "location": "Iztapalapa", "room_type": "Private room", "accommodates": null, "bedrooms": 1, "beds": null, "bathrooms": null, "parking": null, "patio_balcon": null}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Iztapalapa  [alias]
  Tipo de alojamiento   : Private room
  Latitud               : 19.3566
  Longitud              : -99.0801
  Ocupantes             : 2
  Habitaciones          : 1
  Camas                 : 1
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      438.64 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Iztapalapa',
  'room_type': 'Private room',
  'latitude': 19.3566,
  'longitude': -99.0801,
  'accommodates': 2,
  'bedrooms': 1,
  'beds': 1,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 438.64}

In [ ]:
pipeline.predict_from_text('tengo un departamento completo en Polanco para hasta 6 personas, con 4 recámaras, 3.5 baños, estacionamiento y balcón')

2026-05-22 07:17:51 | INFO     | bedrock:bedrock | Solicitud recibida: 'tengo un departamento completo en Polanco para hasta 6 personas, con 4 recámaras, 3.5 baños, estacio'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:17:52 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Polanco", "room_type": "Entire home/apt", "accommodates": 6, "bedrooms": 4, "beds": null, "bathrooms": 3.5, "parking": 1, "patio_balcon": 1}


2026-05-22 07:17:52 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Miguel Hidalgo", "room_type": "Entire home/apt", "latitude": 19.432, "longitude": -99.1914, "accommodates": 6, "bedrooms": 4, "beds": 4, "bathrooms": 3.5, "parking": 1, "patio_balcon": 1}


2026-05-22 07:17:52 | INFO     | bedrock:bedrock | Predicción exitosa: $3,686.42 MXN | neighbourhood=Miguel Hidalgo | room_type=Entire home/apt


   JSON extraído: {"is_valid_listing": true, "location": "Polanco", "room_type": "Entire home/apt", "accommodates": 6, "bedrooms": 4, "beds": null, "bathrooms": 3.5, "parking": 1, "patio_balcon": 1}
[2/3] Resolviendo ubicación
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Miguel Hidalgo  [alias]
  Tipo de alojamiento   : Entire home/apt
  Latitud               : 19.4320
  Longitud              : -99.1914
  Ocupantes             : 6
  Habitaciones          : 4
  Camas                 : 4
  Baños                 : 3.5
  Estacionamiento       : ✓ Sí
  Patio/Balcón          : ✓ Sí
──────────────────────────────────────────────────────
Precio estimado : $    3,686.42 MXN / noche
──────────────────────────────────────────────────────



{'fields': {'neighbourhood': 'Miguel Hidalgo',
  'room_type': 'Entire home/apt',
  'latitude': 19.432,
  'longitude': -99.1914,
  'accommodates': 6,
  'bedrooms': 4,
  'beds': 4,
  'bathrooms': 3.5,
  'parking': 1,
  'patio_balcon': 1},
 'price': 3686.42}

In [ ]:
pipeline.predict_from_text('Tengo un departamento en la calle de Cerro del vigilante 91, para 5 personas, 2 habitaciones, sin estacionamiento')

2026-05-22 07:17:59 | INFO     | bedrock:bedrock | Solicitud recibida: 'Tengo un departamento en la calle de Cerro del vigilante 91, para 5 personas, 2 habitaciones, sin es'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:18:00 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Cerro del Vigilante 91", "room_type": "Entire home/apt", "accommodates": 5, "bedrooms": 2, "beds": null, "bathrooms": null, "parking": 0, "patio_balcon": null}


2026-05-22 07:18:00 | INFO     | bedrock:bedrock | Caché HIT para 'Cerro del Vigilante 91' (match='Cerro del Vigilante 91', score=1.00): Coyoacán (lat=19.3350, lon=-99.1620)


2026-05-22 07:18:00 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Coyoacán", "room_type": "Entire home/apt", "latitude": 19.335, "longitude": -99.162, "accommodates": 5, "bedrooms": 2, "beds": 2, "bathrooms": 1.0, "parking": 0, "patio_balcon": 0}


2026-05-22 07:18:00 | INFO     | bedrock:bedrock | Predicción exitosa: $850.11 MXN | neighbourhood=Coyoacán | room_type=Entire home/apt


   JSON extraído: {"is_valid_listing": true, "location": "Cerro del Vigilante 91", "room_type": "Entire home/apt", "accommodates": 5, "bedrooms": 2, "beds": null, "bathrooms": null, "parking": 0, "patio_balcon": null}
[2/3] Resolviendo ubicación
Caché HIT [exacto]: 'Cerro del Vigilante 91' → 'Cerro del Vigilante 91' → Coyoacán (lat=19.3350, lon=-99.1620)
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Coyoacán  [geocodificado por Bedrock]
  Tipo de alojamiento   : Entire home/apt
  Latitud               : 19.3350
  Longitud              : -99.1620
  Ocupantes             : 5
  Habitaciones          : 2
  Camas                 : 2
  Baños                 : 1.0
  Estacionamiento       : ✗ No
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $      850.11 MXN / noche
───────

{'fields': {'neighbourhood': 'Coyoacán',
  'room_type': 'Entire home/apt',
  'latitude': 19.335,
  'longitude': -99.162,
  'accommodates': 5,
  'bedrooms': 2,
  'beds': 2,
  'bathrooms': 1.0,
  'parking': 0,
  'patio_balcon': 0},
 'price': 850.11}

In [ ]:
pipeline.predict_from_text(
    "Tengo una habitación privada en Plaza Necaxa 17, con 2 camas, para 4 personas y estacionamiento."
)

2026-05-22 07:18:23 | INFO     | bedrock:bedrock | Solicitud recibida: 'Tengo una habitación privada en Plaza Necaxa 17, con 2 camas, para 4 personas y estacionamiento.'


[1/3] Extrayendo campos con Bedrock


2026-05-22 07:18:24 | INFO     | bedrock:bedrock | Extracción Bedrock: {"is_valid_listing": true, "location": "Plaza Necaxa 17", "room_type": "Private room", "accommodates": 4, "bedrooms": 1, "beds": 2, "bathrooms": null, "parking": 1, "patio_balcon": null}


2026-05-22 07:18:24 | INFO     | bedrock:bedrock | Caché HIT para 'Plaza Necaxa 17' (match='Plaza Necaxa 17', score=1.00): Cuauhtémoc (lat=19.4312, lon=-99.1389)


2026-05-22 07:18:24 | INFO     | bedrock:bedrock | Campos resueltos: {"neighbourhood": "Cuauhtémoc", "room_type": "Private room", "latitude": 19.4312, "longitude": -99.1389, "accommodates": 4, "bedrooms": 1, "beds": 2, "bathrooms": 1.0, "parking": 1, "patio_balcon": 0}


2026-05-22 07:18:24 | INFO     | bedrock:bedrock | Predicción exitosa: $1,825.55 MXN | neighbourhood=Cuauhtémoc | room_type=Private room


   JSON extraído: {"is_valid_listing": true, "location": "Plaza Necaxa 17", "room_type": "Private room", "accommodates": 4, "bedrooms": 1, "beds": 2, "bathrooms": null, "parking": 1, "patio_balcon": null}
[2/3] Resolviendo ubicación
Caché HIT [exacto]: 'Plaza Necaxa 17' → 'Plaza Necaxa 17' → Cuauhtémoc (lat=19.4312, lon=-99.1389)
[3/3] Calculando precio con SageMaker

──────────────────────────────────────────────────────
Información extraída del alojamiento
──────────────────────────────────────────────────────
  Alcaldía              : Cuauhtémoc  [geocodificado por Bedrock]
  Tipo de alojamiento   : Private room
  Latitud               : 19.4312
  Longitud              : -99.1389
  Ocupantes             : 4
  Habitaciones          : 1
  Camas                 : 2
  Baños                 : 1.0
  Estacionamiento       : ✓ Sí
  Patio/Balcón          : ✗ No
──────────────────────────────────────────────────────
Precio estimado : $    1,825.55 MXN / noche
─────────────────────────────────

{'fields': {'neighbourhood': 'Cuauhtémoc',
  'room_type': 'Private room',
  'latitude': 19.4312,
  'longitude': -99.1389,
  'accommodates': 4,
  'bedrooms': 1,
  'beds': 2,
  'bathrooms': 1.0,
  'parking': 1,
  'patio_balcon': 0},
 'price': 1825.55}

### Agente Strands

Suscripción de correos electrónicos

In [ ]:
import boto3

sns = boto3.client("sns", region_name="us-east-1")

# Crear tópicos
topic_novato  = sns.create_topic(Name="TopicNovato")
topic_experto = sns.create_topic(Name="TopicExperto")

print("Novato ARN: ",  topic_novato["TopicArn"])
print("Experto ARN:", topic_experto["TopicArn"])

# Suscribir correos (AWS enviará un email de confirmación a cada dirección)
sns.subscribe(TopicArn=topic_novato["TopicArn"],  Protocol="email", Endpoint="raulpopoca73@gmail.com")
sns.subscribe(TopicArn=topic_experto["TopicArn"], Protocol="email", Endpoint="dianaba25@gmail.com")

Novato ARN:  arn:aws:sns:us-east-1:141095608224:TopicNovato
Experto ARN: arn:aws:sns:us-east-1:141095608224:TopicExperto


{'SubscriptionArn': 'pending confirmation',
 'ResponseMetadata': {'RequestId': '42ac1abb-6d34-5030-a39e-7832c096663d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '42ac1abb-6d34-5030-a39e-7832c096663d',
   'date': 'Fri, 22 May 2026 01:03:30 GMT',
   'content-type': 'text/xml',
   'content-length': '298',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}

Novato ARN:  arn:aws:sns:us-east-1:141095608224:TopicNovato
Experto ARN: arn:aws:sns:us-east-1:141095608224:TopicExperto


{'SubscriptionArn': 'pending confirmation',
 'ResponseMetadata': {'RequestId': '94849c61-67a5-591c-9cb3-feeb6250e0a8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '94849c61-67a5-591c-9cb3-feeb6250e0a8',
   'date': 'Fri, 22 May 2026 07:20:21 GMT',
   'content-type': 'text/xml',
   'content-length': '298',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}

Novato ARN:  arn:aws:sns:us-east-1:141095608224:TopicNovato
Experto ARN: arn:aws:sns:us-east-1:141095608224:TopicExperto


{'SubscriptionArn': 'pending confirmation',
 'ResponseMetadata': {'RequestId': '34a5abdb-6ab5-5b69-933d-c9516cb58c50',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '34a5abdb-6ab5-5b69-933d-c9516cb58c50',
   'date': 'Fri, 22 May 2026 07:20:32 GMT',
   'content-type': 'text/xml',
   'content-length': '298',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}

La siguiente celda sirve para hacer pruebas

In [ ]:
from agente import procesar_mensaje

casos = [
    "No tengo casa, pero quiero comprar un departamento para invertir",
    "Ya tengo un departamento, quiero información más especializada",
    "¿Cuál es la capital de Francia?", 
]

for caso in casos:
    procesar_mensaje(caso)


Texto recibido: No tengo casa, pero quiero comprar un departamento para invertir



Tool #1: enviar_correo_novato


¡Perfecto! 🏠

He registrado tu interés en comprar un departamento para invertir. Tu consulta ha

 sido enviada a nuestro equipo especializado en prospectos sin propiedad.

Te

 contactaremos pronto con:
- Opciones de departamentos disponibles para inversión
- 

...

porte profesional que necesitas. ¿Hay algún tema específico en el que requieras asesoramiento?
Respuesta del agente:
¡Excelente! 🏢 He registrado tu solicitud de información especializada.

He enviado tu perfil a nuestro equipo de expertos en bienes raíces para clientes con propiedad. Te contactarán con:

✅ Asesoría especializada sobre tu departamento  
✅ Estrategias de optimización y rentabilidad  
✅ Opciones de refinanciamiento o mejoras  
✅ Información sobre inversiones adicionales  
✅ Análisis del mercado inmobiliario actual  

Nuestro equipo te brindará el soporte profesional que necesitas. ¿Hay algún tema específico en el que requieras asesoramiento?

Texto recibido: ¿Cuál es la capital de Francia?


Solo puedo ayudarte con temas de bienes raíces. 🏠

Mi especial

idad es asesorarte sobre compra, venta, inversión y renta

 de inmuebles, casas y departamentos.

¿Hay algo relacionado con bienes raíces en lo que pueda as

istirte?
Respuesta del agente:
Solo puedo ayudarte con temas de bienes raíces. 🏠

Mi especialidad es asesorarte sobre compra, venta, inversión y renta de inmuebles, casas y departamentos.

¿Hay algo relacionado con bienes raíces en lo que pueda asistirte?

